In [1]:
import pandas as pd
import numpy as np

path = r"eth_csv_raw.csv"

df = pd.read_csv(path)
df.head()


,Unnamed: 0,Time(sec),Methane con: (ppm),Ethylene con: (ppm),TGS2602-1,TGS2602-2,TGS2600-1,TGS2600-2,TGS2610-1,TGS2610-2,TGS2620-1,TGS2620-2,TGS2602-3,TGS2602-4,TGS2600-3,TGS2600-4,TGS2610-3,TGS2610-4,TGS2620-3,TGS2620-4
0,0,0.00,0.0,0.0,-41.98,2067.64,-37.13,2.28,8.63,-26.62,-8.46,-0.33,3437.73,2728.14,4054.03,4007.89,4478.27,5056.98,3639.09,3128.49
1,1,0.01,0.0,0.0,-46.50,2067.88,-28.56,13.69,-12.35,-25.81,-5.04,-5.04,3432.44,2734.47,4038.62,4019.40,4496.72,5051.81,3636.97,3115.03
2,2,0.02,0.0,0.0,-36.16,2055.81,-10.89,8.63,-2.93,-30.34,-9.27,-2.12,3438.61,2719.97,4030.92,4025.48,4489.54,5057.35,3641.81,3105.24
3,3,0.03,0.0,0.0,-50.36,2053.68,-31.96,-0.65,-8.29,-21.60,7.98,2.28,3429.51,2720.50,4040.22,4000.87,4485.44,5049.60,3642.72,3124.84
4,4,0.04,0.0,0.0,-37.30,2081.17,-36.16,3.26,5.05,-26.14,-7.48,-0.65,3436.85,2719.71,4029.64,4007.25,4499.12,5057.35,3674.30,3147.59


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4178504 entries, 0 to 4178503
Data columns (total 20 columns):
 #   Column               Dtype  
---  ------               -----  
 0   Unnamed: 0           int64  
 1   Time(sec)            float64
 2   Methane con: (ppm)   float64
 3   Ethylene con: (ppm)  float64
 4   TGS2602-1            float64
 5   TGS2602-2            float64
 6   TGS2600-1            float64
 7   TGS2600-2            float64
 8   TGS2610-1            float64
 9   TGS2610-2            float64
 10  TGS2620-1            float64
 11  TGS2620-2            float64
 12  TGS2602-3            float64
 13  TGS2602-4            float64
 14  TGS2600-3            float64
 15  TGS2600-4            float64
 16  TGS2610-3            float64
 17  TGS2610-4            float64
 18  TGS2620-3            float64
 19  TGS2620-4            float64
dtypes: float64(19), int64(1)
memory usage: 637.6 MB


In [3]:
#Dropping Samples with zero ethylene concentration
df.drop(df[df['Ethylene con: (ppm)'] == 0].index, inplace=True)

In [4]:
#Min max normalization
c = [
 'TGS2602-1',
 'TGS2602-2',
 'TGS2600-1',
 'TGS2600-2',
 'TGS2610-1',
 'TGS2610-2',
 'TGS2620-1',
 'TGS2620-2',
 'TGS2602-3',
 'TGS2602-4',
 'TGS2600-3',
 'TGS2600-4',
 'TGS2610-3',
 'TGS2610-4',
 'TGS2620-3',
 'TGS2620-4']

for i in c:
    df[c] = (df[c]-df[c].min()) / (df[c].max() - df[c].min())
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1802960 entries, 19974 to 4107083
Data columns (total 20 columns):
 #   Column               Dtype  
---  ------               -----  
 0   Unnamed: 0           int64  
 1   Time(sec)            float64
 2   Methane con: (ppm)   float64
 3   Ethylene con: (ppm)  float64
 4   TGS2602-1            float64
 5   TGS2602-2            float64
 6   TGS2600-1            float64
 7   TGS2600-2            float64
 8   TGS2610-1            float64
 9   TGS2610-2            float64
 10  TGS2620-1            float64
 11  TGS2620-2            float64
 12  TGS2602-3            float64
 13  TGS2602-4            float64
 14  TGS2600-3            float64
 15  TGS2600-4            float64
 16  TGS2610-3            float64
 17  TGS2610-4            float64
 18  TGS2620-3            float64
 19  TGS2620-4            float64
dtypes: float64(19), int64(1)
memory usage: 288.9 MB


In [5]:
#Function that remove samples which are only in Interquartile Range (Grouped by Ethylene Concentration)
import pandas as pd


def drop_outliers_by_group(df, column_names):
  def g(df):
    for col in column_names:
      q1 = df[col].quantile(0.25)
      q3 = df[col].quantile(0.75)
      iqr = q3 - q1
      condition = (df[col] >= (q1 - 1.5 * iqr)) & (df[col] <= (q3 + 1.5 * iqr))
      df = df[condition]  # Filter rows based on IQR condition
    return df

  return df.groupby('Ethylene con: (ppm)').apply(g)


In [6]:
#Implementing Above Function
result_df = drop_outliers_by_group(df.copy(), c)

result_df.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 1265340 entries, (np.float64(6.67), np.int64(487264)) to (np.float64(20.0), np.int64(2281375))
Data columns (total 20 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   Unnamed: 0           1265340 non-null  int64  
 1   Time(sec)            1265340 non-null  float64
 2   Methane con: (ppm)   1265340 non-null  float64
 3   Ethylene con: (ppm)  1265340 non-null  float64
 4   TGS2602-1            1265340 non-null  float64
 5   TGS2602-2            1265340 non-null  float64
 6   TGS2600-1            1265340 non-null  float64
 7   TGS2600-2            1265340 non-null  float64
 8   TGS2610-1            1265340 non-null  float64
 9   TGS2610-2            1265340 non-null  float64
 10  TGS2620-1            1265340 non-null  float64
 11  TGS2620-2            1265340 non-null  float64
 12  TGS2602-3            1265340 non-null  float64
 13  TGS2602-4            1265340 non-null

C:\Users\Manjunadh\AppData\Local\Temp\ipykernel_13580\1159419852.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('Ethylene con: (ppm)').apply(g)


In [7]:
result_df = result_df[1:]
result_df.to_csv(r"Removed_Outliers.csv")